# Milestone 2 - Week 1: Data and Baseline Foundation
## (pre-labeled Hugging Face maze-curriculum dataset)

Rebuilt on **`selimaktas/maze-curriculum-dataset`** (Hugging Face, MIT license,
60k ASCII mazes with provided solutions). The labelling question is decided
exactly as demanded: **no BFS and no other search anywhere in the data
pipeline** - the label masks are replayed from the *provided* solution strings
by pure rule-following. Search appears only in Section 4, as an *evaluation*
tool for predictions, never to create labels.

This notebook is **self-contained** (runs in Colab or locally with no repo
files next to it): models are defined inline; the only dependencies are
`torch`, `numpy`, `matplotlib` and `datasets`.

Week-1 checklist:

- [x] **Dataset selection** - a documented existing dataset (Section 1)
- [x] **Data adapter** - ASCII mazes -> 3-channel tensors, provided solutions -> masks (Section 2)
- [x] **Dataset validation** - endpoints, legality of every provided solution, duplicates (Section 3)
- [x] **Evaluation** - strict legal-path metric instead of connectivity alone (Section 4)
- [x] **First baseline** - trained RecurrentCNN (Section 5, results Section 6)
- [x] **P1: simple CNN** - non-recurrent control (Section 7)
- [x] **Hyperparameters, numerical results, visual comparisons, limitations** (Sections 5-8)


In [ ]:
# ============================ 0. Setup =========================
%matplotlib inline
import json
import sys
import time
from collections import Counter, deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# hint: 'datasets' (Hugging Face) is auto-installed here if missing
try:
    import datasets
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"],
                   check=False)
    import datasets

DEVICE = torch.device("cpu")          # Week 1 runs comfortably on CPU
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS = Path.cwd() / "week1_results"
RESULTS.mkdir(exist_ok=True)

print("torch", torch.__version__, "| numpy", np.__version__,
      "| datasets", datasets.__version__, "| device:", DEVICE)


## 1) Dataset selection (P0)

**Chosen dataset: `selimaktas/maze-curriculum-dataset`** (Hugging Face, MIT
license). Why this one:

- a genuinely **pre-labeled, existing** dataset - solutions are *provided*
  (move strings), so the labels are not created by our own solver;
- `60,000` mazes with a built-in curriculum: grid_size 5 (25k), 7 (20k), 9 (15k)
  navigable cells - Week 1 uses the easy 5x5 level, later weeks scale up;
- per the dataset card the mazes are generated with Wilson's algorithm, i.e.
  **perfect mazes with exactly one solution path** - labels are unambiguous;
- compact (5.4 MB, parquet) and trivially re-downloadable - reproducible.

Fields per instance:

| field | type | content |
|---|---|---|
| `prompt` | string | LLM instruction text (unused here - we work on tensors) |
| `maze` | string | ASCII maze: `#` wall, ` ` free, `S` start, `E` end |
| `solution` | string | provided solution as moves N/S/E/W |
| `grid_size` | int | navigable cells per axis; ASCII grid is `2N+1` per side |

ASCII-to-tensor geometry: one move of the solution = **2 ASCII cells**
(landing cell + the wall-gap between). The Week-1 level (grid_size 5) is an
`11 x 11` ASCII grid.

Work split: the dataset ships only `train` - we carve our own
train / val / test split with a fixed seed (Section 3).


In [ ]:
# ======================= 1. Load and document the dataset =======================
CONFIG = dict(
    dataset="selimaktas/maze-curriculum-dataset (Hugging Face, MIT license)",
    week1_subset="grid_size = 5  ->  ASCII 11x11 (the easy curriculum level)",
    n_train=4000, n_val=1000, n_test=1000, seed=0,
)
print(json.dumps(CONFIG, indent=2))

raw = datasets.load_dataset("selimaktas/maze-curriculum-dataset")["train"]
print("columns:", raw.column_names, "| rows:", len(raw))
print("curriculum levels (grid_size -> count):", dict(Counter(raw["grid_size"])))

ex = raw[0]
print("\nprompt[:52]:", ex["prompt"][:52])
print("grid_size:", ex["grid_size"], "| provided solution:", repr(ex["solution"]))
print("the ASCII maze (S = start, E = end):")
print(ex["maze"])

moves = [len(raw[i]["solution"]) for i in range(4000)]
moves = np.array(moves)
print("solution length on a 4k sample: mean %.1f | max %d moves"
      % (moves.mean(), moves.max()))


## 2) Data adapter (P0)

Two pure functions, **no search of any kind**:

- `parse_maze` - ASCII -> wall grid + start/goal coordinates + 3-channel tensor
  `[walkable, start one-hot, goal one-hot]`;
- `replay_solution` - the **provided** move string is literally replayed from
  the start cell: every move must stay in bounds, not land on or pass through a
  wall, and the walk must end exactly at `E`. Every traversed ASCII cell
  (landing cells and wall-gap cells) is marked -> the solution mask `y`.

Label semantics: for these perfect mazes the unique S->E path *is* the mask.


In [ ]:
# ======================= 2. Adapter: ASCII -> tensors, solution -> mask =======================
DELTAS = {"N": (-2, 0), "S": (2, 0), "E": (0, 2), "W": (0, -2)}


def parse_maze(maze_str):
    # ASCII maze -> wall grid (1 = wall), start and goal coordinates
    lines = [ln for ln in maze_str.split("\n") if ln.strip()]
    H, W = len(lines), len(lines[0])
    wall = np.zeros((H, W), np.int8)
    start = goal = None
    for r, line in enumerate(lines):
        for c, ch in enumerate(line):
            if ch == "#":
                wall[r, c] = 1
            elif ch == "S":
                start = (r, c)
            elif ch == "E":
                goal = (r, c)
    return wall, start, goal


def maze_to_channels(wall, start, goal):
    # wall grid + endpoints -> model input tensor (3, H, W)
    walk = (wall == 0).astype(np.float32)
    s = np.zeros(wall.shape, np.float32); s[start] = 1.0
    g = np.zeros(wall.shape, np.float32); g[goal] = 1.0
    return np.stack([walk, s, g])


def replay_solution(wall, start, solution):
    # PROVIDED solution string -> path mask by pure rule-following (no search).
    # Marks every traversed ASCII cell (landing + gap). Returns (mask, legal).
    H, W = wall.shape
    mask = np.zeros((H, W), bool)
    mask[start] = True
    r, c = start
    for mv in solution:
        dr, dc = DELTAS[mv]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < H and 0 <= nc < W):
            return mask, False
        if wall[r + dr // 2, c + dc // 2] or wall[nr, nc]:
            return mask, False
        mask[r + dr // 2, c + dc // 2] = True
        mask[nr, nc] = True
        r, c = nr, nc
    return mask, True


wall, start, goal = parse_maze(ex["maze"])
assert start is not None and goal is not None
assert wall[start] == 0 and wall[goal] == 0
x_demo = maze_to_channels(wall, start, goal)
mask_demo, ok = replay_solution(wall, start, ex["solution"])
assert ok, "the provided solution must replay legally on a correct dataset"
assert mask_demo[goal], "the replayed path must end on E"
assert not (mask_demo & (wall == 1)).any(), "the path must not touch walls"
print("adapter demo OK | x:", x_demo.shape, "| path cells in y:", int(mask_demo.sum()),
      "| moves:", len(ex["solution"]))

fig, axes = plt.subplots(1, 4, figsize=(11, 3.0))
for ax, ch, ttl in zip(axes, x_demo,
                       ["walkable channel", "start channel", "goal channel", "solution mask y"]):
    ax.imshow(ch, cmap="Greens" if "mask" in ttl else "gray", vmin=0, vmax=1.3)
    ax.set_title(ttl, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()


## 3) Dataset validation (P0)

Checks enforced on **every kept sample** - all by rule-following, no search:

1. the ASCII maze contains exactly one `S` and one `E`, both on free cells;
2. the **provided** solution replays legally: every move in bounds, neither the
   gap nor the landing cell is a wall, and the walk ends exactly on `E`;
3. the replayed mask never covers a wall cell;
4. no duplicate mazes (hash of the ASCII string).

Samples failing any check are dropped and counted (the report cell prints the
drop rate). Validation also produces the rotated split: a seeded random
subsample of the valid pool becomes train / val / test.


In [ ]:
# ======================= 3. Validate every provided solution; build the split =======================
def validate_ex(maze_str, solution):
    issues = []
    wall, start, goal = parse_maze(maze_str)
    counts = maze_str.count("S"), maze_str.count("E")
    if counts != (1, 1):
        return wall, None, None, ["must contain exactly one S and one E"]
    if wall[start] == 1 or wall[goal] == 1:
        issues.append("endpoint on a wall")
    mask, ok = replay_solution(wall, start, solution)
    if not ok:
        issues.append("provided solution is not a legal walk")
    elif not mask[goal]:
        issues.append("replayed path does not end on E")
    elif (mask & (wall == 1)).any():
        issues.append("path covers a wall cell")
    return wall, start, goal, issues


pool = [i for i, g in enumerate(raw["grid_size"]) if g == 5]
print("size-5 pool:", len(pool), "mazes")

kept, dropped, dups, seen = [], [], 0, set()
t0 = time.time()
for i in pool:
    maze_str = raw[i]["maze"]
    key = maze_str
    if key in seen:
        dups += 1
        continue
    seen.add(key)
    wall, start, goal, issues = validate_ex(maze_str, raw[i]["solution"])
    if issues:
        dropped.append((i, issues))
    else:
        kept.append(i)
print("validated %d mazes in %.1fs" % (len(pool), time.time() - t0))
print("kept: %d | dropped: %d | duplicate mazes: %d" % (len(kept), len(dropped), dups))
if dropped:
    print("first dropped:", dropped[:3])

rng = np.random.default_rng(CONFIG["seed"])
perm = rng.permutation(len(kept))
n_tr, n_va, n_te = CONFIG["n_train"], CONFIG["n_val"], CONFIG["n_test"]
pick = [kept[j] for j in perm[: n_tr + n_va + n_te]]
splits = {"train": pick[:n_tr], "val": pick[n_tr:n_tr + n_va], "test": pick[n_tr + n_va:]}


def build_tensors(indices):
    xs, ys, walls, starts, goals, sols = [], [], [], [], [], []
    for i in indices:
        maze_str, solution = raw[i]["maze"], raw[i]["solution"]
        wall, start, goal = parse_maze(maze_str)
        mask, ok = replay_solution(wall, start, solution)
        assert ok  # validated above; a crash here means a bug, not bad data
        xs.append(maze_to_channels(wall, start, goal))
        ys.append(mask.astype(np.float32))
        walls.append(wall)
        starts.append(start)
        goals.append(goal)
        sols.append(solution)
    return (torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(ys)),
            walls, starts, goals, sols)


data = {name: build_tensors(idx) for name, idx in splits.items()}
for name, (x, y, *_rest) in data.items():
    print("%-5s tensors: x %s %s | y %s %s" %
          (name, tuple(x.shape), x.dtype, tuple(y.shape), y.dtype))

pos_weight = float((1.0 - data["train"][1].mean().item()) / data["train"][1].mean().item())
print("pos_weight (train) = %.2f" % pos_weight)

# dataset statistics on the training split
lens = np.array([len(s) for s in data["train"][5]])
cells = np.array([float(yk.sum()) for yk in data["train"][1]])
free = np.array([float((xk[0] > 0.5).sum()) for xk in data["train"][0]])
print("train: solution moves mean %.1f | max %d" % (lens.mean(), lens.max()))
print("train: path cells per mask: mean %.1f (of %.1f free cells)" % (cells.mean(), free.mean()))
print("train: label density mean = %.3f (sparse positive class)"
      % (cells / free).mean())
